In [ ]:
import os, sys

IS_COLAB = 'google.colab' in sys.modules or os.path.exists('/content')

if IS_COLAB:
    # Pin compatible versions and upgrade torchao to fix PEFT compatibility
    os.system('pip install -q transformers==4.44.0 datasets==2.19.0 peft==0.12.0 accelerate==0.33.0 torchao==0.16.0 python-dotenv')
    os.system('nvidia-smi')
    print('Packages installed. IMPORTANT: Go to Runtime → Restart session, then re-run all cells.')
else:
    print('Running locally — skipping pip install and nvidia-smi')


In [15]:
from google.colab import drive
drive.mount('/content/drive')

# Option A: Upload unified_biology_dataset.json manually to /content/data/unified_biology_dataset.json
# Option B: Clone repo from GitHub
# !git clone https://github.com/<your-repo>/EduHinglish.git

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [16]:
import json
import random
from pathlib import Path
from collections import Counter
from transformers import AutoTokenizer
from datasets import Dataset, DatasetDict

# Path to the dataset in Google Drive
DATASET_PATH = Path("/content/drive/MyDrive/EduHinglish/data/unified_biology_dataset.json")
LID_OUTPUT_DIR = Path("/content/muril_lid_dataset")
MODEL_NAME = "google/muril-base-cased"
MAX_LENGTH = 128
SEED = 42

LID_LABEL2ID = {"HI": 0, "EN": 1, "NE": 2, "UNIV": 3, "MIX": 4}
LID_ID2LABEL = {v: k for k, v in LID_LABEL2ID.items()}
LID_LABELS = list(LID_LABEL2ID.keys())

_PUNCT_CHARS = frozenset(list(".,?!;:()-/") + ['"', "'", "–", "—", "…"])
_PUNCT_STRIP = "".join(_PUNCT_CHARS)

def strip_punct(word: str) -> str: return word.strip(_PUNCT_STRIP)
def is_punct_token(text: str) -> bool: return len(text) > 0 and all(ch in _PUNCT_CHARS for ch in text)

def lookup_label(word, labels_dict):
    cleaned = strip_punct(word)
    lbl = labels_dict.get(word) or labels_dict.get(word.lower()) or labels_dict.get(word.capitalize())
    if lbl: return lbl
    if cleaned != word:
        lbl = labels_dict.get(cleaned) or labels_dict.get(cleaned.lower()) or labels_dict.get(cleaned.capitalize())
        if lbl: return lbl
    if cleaned:
        cleaned_lower = cleaned.lower()
        for k, v in labels_dict.items():
            if strip_punct(k).lower() == cleaned_lower: return v
    if is_punct_token(word) or not cleaned: return "UNIV"
    return "EN"

print("Loading dataset...")
with open(DATASET_PATH, encoding="utf-8") as f:
    dataset = json.load(f)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
all_examples = []
label_counter = Counter()

for entry in dataset:
    hinglish = entry.get("hinglish_roman", "")
    labels_dict = entry.get("word_level_labels", {})
    if not hinglish or not labels_dict: continue
    
    words = hinglish.split()
    word_labels = [lookup_label(w, labels_dict) if lookup_label(w, labels_dict) in LID_LABEL2ID else "EN" for w in words]
    
    tokenized = tokenizer(words, is_split_into_words=True, max_length=MAX_LENGTH, truncation=True, padding=False)
    word_ids = tokenized.word_ids()
    
    aligned_labels = []
    previous_word_id = None
    for wid in word_ids:
        if wid is None: aligned_labels.append(-100)
        elif wid != previous_word_id:
            lbl_str = word_labels[wid]
            aligned_labels.append(LID_LABEL2ID[lbl_str])
            label_counter[lbl_str] += 1
        else: aligned_labels.append(-100)
        previous_word_id = wid
        
    all_examples.append({
        "input_ids": tokenized["input_ids"],
        "attention_mask": tokenized["attention_mask"],
        "labels": aligned_labels,
    })

random.seed(SEED)
random.shuffle(all_examples)
cut = max(1, int(len(all_examples) * 0.8))
train_examples, dev_examples = all_examples[:cut], all_examples[cut:]

def _to_dataset(examples):
    return Dataset.from_dict({
        "input_ids": [e["input_ids"] for e in examples],
        "attention_mask": [e["attention_mask"] for e in examples],
        "labels": [e["labels"] for e in examples]
    })

ds = DatasetDict({"train": _to_dataset(train_examples), "dev": _to_dataset(dev_examples)})
LID_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
ds.save_to_disk(str(LID_OUTPUT_DIR))
print(f"Prepared LID data saved to {LID_OUTPUT_DIR}")

Loading dataset...


Saving the dataset (0/1 shards):   0%|          | 0/760 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/190 [00:00<?, ? examples/s]

Prepared LID data saved to /content/muril_lid_dataset


In [ ]:
import torch
import numpy as np
from transformers import AutoModelForTokenClassification, DataCollatorForTokenClassification, Trainer, TrainingArguments
from peft import LoraConfig, TaskType, get_peft_model
from datasets import load_from_disk

ds = load_from_disk(str(LID_OUTPUT_DIR))
train_ds, dev_ds = ds["train"], ds["dev"]

model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME, num_labels=len(LID_LABEL2ID), id2label=LID_ID2LABEL, label2id=LID_LABEL2ID
)

lora_config = LoraConfig(
    task_type=TaskType.TOKEN_CLS, r=16, lora_alpha=32, lora_dropout=0.1, bias="none",
    target_modules=["query", "key", "value", "dense"]
)
model = get_peft_model(model, lora_config)

training_args = TrainingArguments(
    output_dir="/content/muril_lid_checkpoints",
    num_train_epochs=10,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-4,
    weight_decay=0.01,
    warmup_ratio=0.1,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    logging_steps=10,
    fp16=torch.cuda.is_available(),
    report_to="none",
    save_total_limit=2,
)

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer, padding=True, max_length=128)

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    preds = np.argmax(predictions, axis=-1)
    
    true_labels, pred_labels = [], []
    for pred_seq, label_seq in zip(preds, labels):
        for p, l in zip(pred_seq, label_seq):
            if l != -100:
                true_labels.append(l)
                pred_labels.append(p)
                
    true_labels = np.array(true_labels)
    pred_labels = np.array(pred_labels)
    accuracy = (true_labels == pred_labels).mean() * 100
    
    label_metrics = {}
    for label_id in range(len(LID_LABEL2ID)):
        label_name = LID_ID2LABEL[label_id]
        tp = int(((pred_labels == label_id) & (true_labels == label_id)).sum())
        fp = int(((pred_labels == label_id) & (true_labels != label_id)).sum())
        fn = int(((pred_labels != label_id) & (true_labels == label_id)).sum())
        
        precision = (tp / (tp + fp) * 100) if (tp + fp) > 0 else 0.0
        recall    = (tp / (tp + fn) * 100) if (tp + fn) > 0 else 0.0
        f1        = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0.0
        
        label_metrics[label_name] = {"precision": precision, "recall": recall, "f1": f1, "support": tp + fn}
        
    return {"accuracy": accuracy, "label_metrics": label_metrics}

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=dev_ds,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [ ]:
trainer.train()

In [ ]:
eval_result = trainer.evaluate()
print(f"Accuracy: {eval_result['eval_accuracy']:.2f}%")

label_metrics = eval_result.get("eval_label_metrics", {})
print("Per-label metrics:")
print(f"{'Label':>6s} | {'Precision':>10s} | {'Recall':>10s} | {'F1':>10s} | {'Support':>8s}")
for label_name in ["HI", "EN", "NE", "UNIV", "MIX"]:
    m = label_metrics.get(label_name, {})
    if m:
        print(f"{label_name:>6s} | {m.get('precision', 0):>9.1f}% | {m.get('recall', 0):>9.1f}% | {m.get('f1', 0):>9.1f}% | {m.get('support', 0):>8d}")

In [ ]:
import os
# Merge LoRA weights into base model before saving
merged_model = model.merge_and_unload()
# Fix non-contiguous tensors after LoRA merge
for param in merged_model.parameters():
    param.data = param.data.contiguous()
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/EduHinglish/models/muril_lid_v1"
os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)

merged_model.save_pretrained(DRIVE_OUTPUT_DIR)
tokenizer.save_pretrained(DRIVE_OUTPUT_DIR)
print(f"Model saved to {DRIVE_OUTPUT_DIR}")

In [ ]:
import re

LID_TESTS = [
    "Mitochondria ko cell ka powerhouse kehte hain.",
    "Sir, photosynthesis kaise hoti hai?",
    "DNA replication mein enzyme kya role play karta hai?",
    "Mendel ne pea plants par experiments kiye the.",
    "Yeh process anaerobic respiration kehlata hai.",
]

def predict_lid(text: str, model, tokenizer):
    words = re.findall(r"\w+|[^\w\s]", text)
    encoding = tokenizer(words, is_split_into_words=True, return_tensors="pt", truncation=True).to(model.device)
    
    with torch.no_grad():
        outputs = model(**encoding)
        predictions = torch.argmax(outputs.logits, dim=-1).squeeze(0).tolist()
        
    word_ids = encoding.word_ids()
    result = []
    current_word_idx = None
    
    for idx, word_idx in enumerate(word_ids):
        if word_idx is None: continue
        if word_idx != current_word_idx:
            current_word_idx = word_idx
            pred_label = model.config.id2label.get(predictions[idx], "O")
            result.append((words[word_idx], pred_label))
    return result

print("── MuRIL LID Predictions ──")
for sent in LID_TESTS:
    print(f'Input: "{sent}"')
    preds = predict_lid(sent, merged_model, tokenizer)
    for word, label in preds:
        spacing = "  " if len(label) == 2 else " "
        print(f"  {word:<12} → {label}{spacing}")
    print()